# `llm-synth` walkthrough: synthetic diabetes data, end to end

This notebook walks through the full `llm-synth` pipeline stage by stage, using the bundled
500-row diabetes sample (`data/samples/diabetes_sample.csv` — UCI "Diabetes 130-US hospitals",
public domain) and the pre-configured `domains/diabetes.yaml` domain config.

Pipeline stages, in order:

1. **profile** — extract per-column statistics from the seed CSV
2. **dp** — apply differential privacy (Laplace noise) to those statistics
3. **generate** — prompt an LLM with the DP statistics to produce synthetic rows *(requires `OPENAI_API_KEY`, makes paid API calls)*
4. **validate** — check the synthetic data's schema, distribution fidelity, correlation fidelity, privacy, and clinical plausibility
5. **tstr** — Train-on-Synthetic / Test-on-Real: how well a model trained on synthetic data predicts on real held-out data

> **Cost note:** Step 3 is the only step that calls the OpenAI API. It's set to generate a small
> number of rows here (`TOTAL_ROWS = 100`) to keep this walkthrough cheap and fast. Increase it
> once you're comfortable with the flow — see `llm-synth run --rows <N>` for full runs.

## Setup

Run this notebook from the **project root** (the directory containing `config.yaml`,
`domains/`, `prompts/`, and `data/`) — that's how `llm-synth` locates its configuration
(see `llm_synth.config.get_root`). If you've installed `llm-synth` as a package elsewhere,
either run the notebook from this project directory or set `LLMSYNTH_ROOT` to point at it.

You'll also need `OPENAI_API_KEY` set (e.g. in a `.env` file at the project root, or exported
in your shell) before running Step 3.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

from llm_synth.config import get_root, set_domain, load_config

ROOT = get_root()
print("Project root:", ROOT)

CONDITION = "diabetes"
SLUG = CONDITION
SEED_CSV = ROOT / "data" / "samples" / "diabetes_sample.csv"

OUTPUT_DIR = ROOT / "processed_data" / SLUG
STAT_PROFILE_DIR = OUTPUT_DIR / "stat_profile"
for d in (OUTPUT_DIR, STAT_PROFILE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Activate the diabetes domain config (schema, TSTR target, clinical-plausibility rules).
# This must happen before importing generate/validate/tstr, which read config at import time —
# safe here since we haven't imported them yet.
set_domain(ROOT / "domains" / f"{CONDITION}.yaml")

print("Seed CSV:        ", SEED_CSV)
print("Output directory:", OUTPUT_DIR)

Quick look at the seed data we're working with:

In [ ]:
seed_df = pd.read_csv(SEED_CSV)
print(f"{len(seed_df):,} rows × {len(seed_df.columns)} columns")
seed_df.head()

---
## Step 1 — Profile: extract seed statistics

`extract_seed_stats` computes per-column distributions, correlations, and missingness rates
from the seed CSV. This is the *only* place real seed data is read — its output (aggregate
statistics, not raw rows) is what downstream stages and the LLM ever see.

In [ ]:
from llm_synth.seed_statistics import extract_seed_stats

stats_path = STAT_PROFILE_DIR / f"{SLUG}_stats.json"
seed_stats = extract_seed_stats(csv_path=str(SEED_CSV), output_json=str(stats_path))

print(f"Profiled {seed_stats['meta']['n_rows']:,} rows, {seed_stats['meta']['n_columns']} columns")
print("Saved to:", stats_path)
print("Top-level sections:", list(seed_stats.keys()))

---
## Step 2 — Apply differential privacy

`apply_dp` adds calibrated Laplace noise to the seed statistics before they're ever shown to
the LLM. The privacy budget `epsilon` controls the noise/utility trade-off — lower epsilon
means stronger privacy guarantees but noisier statistics (and therefore less faithful synthetic
data). `1.0` is a reasonable default to start with.

In [ ]:
from llm_synth.dp_statistics import apply_dp

EPSILON = 1.0

dp_stats = apply_dp(seed_stats, epsilon=EPSILON, seed=42)
dp_stats_path = STAT_PROFILE_DIR / f"{SLUG}_dp_stats.json"
dp_stats_path.write_text(json.dumps(dp_stats, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"DP statistics (ε = {EPSILON}) saved to:", dp_stats_path)

---
## Step 3 — Generate synthetic data

`generate_synthetic_data` sends the DP statistics — never raw seed rows — to an LLM in
batches, along with a system prompt describing the domain (`prompts/system_prompt_diabetes.md`,
see [Prompt Generation](../README.md#prompt-generation) in the main README for how that's built).
Each batch gets a freshly-composed user message built automatically from the statistics.

**This is the only step that calls the OpenAI API and incurs cost.** `TOTAL_ROWS` is kept small
here on purpose — raise it for a real run (`llm-synth generate --rows 5000 ...` from the CLI is
equivalent and supports resuming/parallel batches more conveniently for large runs).

In [ ]:
assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY (e.g. in a .env file at the project root) before running this cell — "
    "this step makes paid calls to the OpenAI API."
)

from llm_synth.generate import generate_synthetic_data

TOTAL_ROWS = 100  # kept small for this walkthrough — raise for a real run

synthetic_csv = OUTPUT_DIR / "synthetic_output.csv"
generate_synthetic_data(
    total_rows=TOTAL_ROWS,
    stats_file=str(dp_stats_path),
    output_csv=str(synthetic_csv),
    seed_csv=str(SEED_CSV),
)

synth_df = pd.read_csv(synthetic_csv)
print(f"Generated {len(synth_df):,} rows → {synthetic_csv}")
synth_df.head()

---
## Step 4 — Validate the synthetic data

`validate_synthetic_data` checks schema conformance, distribution fidelity (vs. the seed
statistics), correlation fidelity, privacy (e.g. exact-row leakage from the seed set), and
clinical plausibility (domain-specific rules from `domains/diabetes.yaml`).

`run_llm=False` skips an additional LLM-based deep-analysis pass (another optional paid call) —
set it to `True` for the full report once you're ready.

In [ ]:
from llm_synth.validate import validate_synthetic_data

validation_report_path = OUTPUT_DIR / f"{SLUG}_validation_report.json"
report = validate_synthetic_data(
    synthetic_csv=str(synthetic_csv),
    seed_stats_json=str(stats_path),
    seed_csv=str(SEED_CSV),
    output_report=str(validation_report_path),
    run_llm=False,  # set True for the full LLM-assisted report (extra paid API calls)
)

print("Validation report saved to:", validation_report_path)
print("Top-level sections:", list(report.keys()))

---
## Step 5 — TSTR: Train-on-Synthetic, Test-on-Real

The TSTR evaluation trains a predictive model on the *synthetic* data and tests it on *real*
held-out seed data, then compares that to a model trained and tested entirely on real data.
The closer the synthetic-trained model's performance is to the real-trained one, the more
useful the synthetic data is as a stand-in for downstream modeling — reported here as an
F1-macro fidelity ratio. Target column and class ordering come from `domains/diabetes.yaml`'s
`tstr` section.

In [ ]:
from llm_synth.pipeline import _run_tstr

tstr_result = _run_tstr(
    seed_file=str(SEED_CSV),
    synthetic_csv=str(synthetic_csv),
    output_dir=OUTPUT_DIR,
    slug=SLUG,
)

tstr_result

---
## Recap: where everything landed

```
processed_data/diabetes/
├── synthetic_output.csv                  ← generated rows (Step 3)
├── diabetes_validation_report.json       ← validation report (Step 4)
├── diabetes_tstr_comparison.json         ← TSTR results (Step 5)
└── stat_profile/
    ├── diabetes_stats.json               ← seed statistics (Step 1)
    └── diabetes_dp_stats.json            ← DP-noised statistics (Step 2)
```

## Next steps

- Run the same flow end-to-end from the CLI: `llm-synth run --seed data/samples/diabetes_sample.csv --condition diabetes --rows 5000`
- Try a different privacy budget by changing `EPSILON` in Step 2 and comparing validation/TSTR results
- Bring your own dataset — see **"Running for a New Disease Area"** in the main [README](../README.md)
- Regenerate or tune the system prompt — see **"Prompt Generation"** in the main [README](../README.md#prompt-generation)